# 05_crash_data_cleaning.ipynb
## Project Title: Traffic Accident Risk Prediction (TARP)

**Unit:** SIT782  
**Prepared by:** Khalid  
**Project Team:** Suba (225094537), Burhan (224802775), Khalid (224696667)  
**Task:** Perform cleaning of the Victorian Road Crash dataset (Week 2 of 8)

### Objectives
- Remove duplicate crash records
- Format crash timestamps
- Clean location variables
- Save a cleaned dataset for further analysis and modelling

## 1. Import libraries

In [23]:
import pandas as pd
import numpy as np

## 2. Load the crash dataset

let's make sure our working directory is up-to-date to avoid any issues accessing different data sets 

In [24]:
import os

os.chdir(r"D:\Git\SIT782\TARP01\MOP-Code\Playground\KhalidT12026\Traffic_Accident_Prediction")

In [25]:
crash_df = pd.read_csv("data/raw/accident.csv")
crash_df.head()

,ACCIDENT_NO,ACCIDENT_DATE,ACCIDENT_TIME,ACCIDENT_TYPE,DAY_OF_WEEK,DCA_CODE,DCA_CODE_DESCRIPTION,LIGHT_CONDITION,POLICE_ATTEND,ROAD_GEOMETRY,...,NO_OF_VEHICLES,HEAVYVEHICLE,PASSENGERVEHICLE,MOTORCYCLE,PT_VEHICLE,DEG_URBAN_NAME,SRNS,RMA,DIVIDED,STAT_DIV_NAME
0,T20230013207,2023-06-06,22:14:00,Collision with vehicle,Tuesday,140,U TURN,Dark Street lights on,Yes,Not at intersection,...,2.0,0.0,2.0,0.0,0.0,MELB_URBAN,NaN,Arterial Other,Undivided,Metro
1,T20240014902,2024-04-19,02:10:00,Collision with vehicle,Friday,140,U TURN,Dark Street lights on,Yes,Not at intersection,...,2.0,0.0,2.0,0.0,0.0,MELB_URBAN,NaN,Local Road,Undivided,Metro
2,T20160009452,2016-04-30,23:50:00,Collision with vehicle,Saturday,120,HEAD ON (NOT OVERTAKING),Dark No street lights,Yes,Not at intersection,...,2.0,0.0,2.0,0.0,0.0,RURAL_VICTORIA,C,Arterial Other,Undivided,Country
3,T20230001223,2023-01-17,15:56:00,Collision with vehicle,Tuesday,113,RIGHT NEAR (INTERSECTIONS ONLY),Day,Yes,Private property,...,2.0,0.0,2.0,0.0,0.0,RURAL_VICTORIA,A,Arterial Highway,Undivided,Country
4,T20220001324,2022-01-21,22:45:00,Fall from or in moving vehicle,Friday,190,FELL IN/FROM VEHICLE,Dark Street lights on,Yes,Cross intersection,...,1.0,0.0,0.0,0.0,0.0,TOWNS,NaN,Local Road,Undivided,Country


## 3. Initial inspection

In [26]:
print("Shape before cleaning:", crash_df.shape)
crash_df.info()

Shape before cleaning: (194437, 52)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 194437 entries, 0 to 194436
Data columns (total 52 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   ACCIDENT_NO             194437 non-null  object 
 1   ACCIDENT_DATE           194437 non-null  object 
 2   ACCIDENT_TIME           194437 non-null  object 
 3   ACCIDENT_TYPE           194437 non-null  object 
 4   DAY_OF_WEEK             194437 non-null  object 
 5   DCA_CODE                194437 non-null  int64  
 6   DCA_CODE_DESCRIPTION    194437 non-null  object 
 7   LIGHT_CONDITION         194437 non-null  object 
 8   POLICE_ATTEND           194437 non-null  object 
 9   ROAD_GEOMETRY           194437 non-null  object 
 10  SEVERITY                194437 non-null  object 
 11  SPEED_ZONE              194437 non-null  object 
 12  RUN_OFFROAD             194437 non-null  object 
 13  ROAD_NAME               194179 non-nul

## 4. Check and remove duplicate records

We first identify exact duplicate rows and then remove them.

In [27]:
duplicate_count = crash_df.duplicated().sum()
print("Number of duplicate rows:", duplicate_count)

Number of duplicate rows: 0


In [28]:
crash_df = crash_df.drop_duplicates()
print("Shape after removing duplicates:", crash_df.shape)

Shape after removing duplicates: (194437, 52)


## 5. Format date and time variables

The original dataset stores date and time as text (`object` type).
We convert them into proper datetime formats so they can be used for analysis and feature engineering.

In [29]:
crash_df["ACCIDENT_DATE"] = pd.to_datetime(crash_df["ACCIDENT_DATE"], errors="coerce")

crash_df["ACCIDENT_TIME"] = pd.to_datetime(
    crash_df["ACCIDENT_TIME"],
    format="%H:%M:%S",
    errors="coerce"
)

print(crash_df[["ACCIDENT_DATE", "ACCIDENT_TIME"]].dtypes)
crash_df[["ACCIDENT_DATE", "ACCIDENT_TIME"]].head()

ACCIDENT_DATE    datetime64[ns]
ACCIDENT_TIME    datetime64[ns]
dtype: object


,ACCIDENT_DATE,ACCIDENT_TIME
0,2023-06-06,1900-01-01 22:14:00
1,2024-04-19,1900-01-01 02:10:00
2,2016-04-30,1900-01-01 23:50:00
3,2023-01-17,1900-01-01 15:56:00
4,2022-01-21,1900-01-01 22:45:00


## 6. Create helpful time-based features

These are not advanced feature engineering yet, but simple cleaned fields that are useful for later EDA and modelling.

In [30]:
crash_df["hour"] = crash_df["ACCIDENT_TIME"].dt.hour
crash_df["day_name"] = crash_df["ACCIDENT_DATE"].dt.day_name()
crash_df["month"] = crash_df["ACCIDENT_DATE"].dt.month
crash_df["year"] = crash_df["ACCIDENT_DATE"].dt.year

crash_df[["ACCIDENT_DATE", "ACCIDENT_TIME", "hour", "day_name", "month", "year"]].head()

,ACCIDENT_DATE,ACCIDENT_TIME,hour,day_name,month,year
0,2023-06-06,1900-01-01 22:14:00,22.0,Tuesday,6,2023
1,2024-04-19,1900-01-01 02:10:00,2.0,Friday,4,2024
2,2016-04-30,1900-01-01 23:50:00,23.0,Saturday,4,2016
3,2023-01-17,1900-01-01 15:56:00,15.0,Tuesday,1,2023
4,2022-01-21,1900-01-01 22:45:00,22.0,Friday,1,2022


## 7. Inspect location variables

For this project, the key location variables are:
- `ROAD_NAME`
- `ROAD_TYPE`
- `LGA_NAME`
- `LATITUDE`
- `LONGITUDE`

We check missing values and basic validity.

In [31]:
location_columns = ["ROAD_NAME", "ROAD_TYPE", "LGA_NAME", "LATITUDE", "LONGITUDE"]
crash_df[location_columns].isnull().sum()

ROAD_NAME     258
ROAD_TYPE    2623
LGA_NAME       90
LATITUDE       85
LONGITUDE      85
dtype: int64

## 8. Clean text-based location variables

We standardise text columns by trimming extra spaces and converting them to uppercase for consistency.

In [32]:
text_location_cols = ["ROAD_NAME", "ROAD_TYPE", "LGA_NAME", "DTP_REGION", "DEG_URBAN_NAME", "STAT_DIV_NAME"]

for col in text_location_cols:
    if col in crash_df.columns:
        crash_df[col] = crash_df[col].astype(str).str.strip().str.upper()

crash_df[text_location_cols].head()

,ROAD_NAME,ROAD_TYPE,LGA_NAME,DTP_REGION,DEG_URBAN_NAME,STAT_DIV_NAME
0,CUMBERLAND,ROAD,MORELAND,INNER METRO,MELB_URBAN,METRO
1,CHAPEL,STREET,STONNINGTON,INNER METRO,MELB_URBAN,METRO
2,BRANDY CREEK,ROAD,BAW BAW,GIPPSLAND,RURAL_VICTORIA,COUNTRY
3,MIDLAND,HIGHWAY,SHEPPARTON,HUME,RURAL_VICTORIA,COUNTRY
4,URQUHART,STREET,MOUNT ALEXANDER,LODDON MALLEE,TOWNS,COUNTRY


## 9. Clean coordinate variables

For mapping and hotspot analysis, valid latitude and longitude values are important.

We:
- remove rows with missing coordinates
- keep only rows with coordinates in valid ranges

In [33]:
print("Missing LATITUDE values:", crash_df["LATITUDE"].isnull().sum())
print("Missing LONGITUDE values:", crash_df["LONGITUDE"].isnull().sum())

Missing LATITUDE values: 85
Missing LONGITUDE values: 85


In [34]:
crash_df = crash_df.dropna(subset=["LATITUDE", "LONGITUDE"])

crash_df = crash_df[
    crash_df["LATITUDE"].between(-90, 90) &
    crash_df["LONGITUDE"].between(-180, 180)
]

print("Shape after cleaning location variables:", crash_df.shape)

Shape after cleaning location variables: (194352, 56)


## 10. Final inspection after cleaning

In [35]:
print("Final shape:", crash_df.shape)
crash_df.info()

Final shape: (194352, 56)
<class 'pandas.core.frame.DataFrame'>
Index: 194352 entries, 0 to 194416
Data columns (total 56 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   ACCIDENT_NO             194352 non-null  object        
 1   ACCIDENT_DATE           194352 non-null  datetime64[ns]
 2   ACCIDENT_TIME           194352 non-null  datetime64[ns]
 3   ACCIDENT_TYPE           194352 non-null  object        
 4   DAY_OF_WEEK             194352 non-null  object        
 5   DCA_CODE                194352 non-null  int64         
 6   DCA_CODE_DESCRIPTION    194352 non-null  object        
 7   LIGHT_CONDITION         194352 non-null  object        
 8   POLICE_ATTEND           194352 non-null  object        
 9   ROAD_GEOMETRY           194352 non-null  object        
 10  SEVERITY                194352 non-null  object        
 11  SPEED_ZONE              194352 non-null  object        
 12  RUN_OFFRO

## 11. Save cleaned dataset

The cleaned dataset is saved for use in future notebooks.

In [36]:
output_path = "data/processed/cleaned_crash_data.csv"
crash_df.to_csv(output_path, index=False)
print(f"Cleaned dataset saved to: {output_path}")

Cleaned dataset saved to: data/processed/cleaned_crash_data.csv


## 12. Summary

In this notebook, the Victorian Road Crash dataset was cleaned by removing duplicate records, converting date and time columns into proper datetime format, and standardising key location variables. Missing and invalid coordinate values were handled to improve data quality for mapping and hotspot analysis. The cleaned dataset was then saved for use in exploratory analysis, feature engineering, and machine learning model development.